# EDA sobre embeddings AlphaEarth Foundations v2.1 (64 dimensiones)

Este notebook evalúa si los embeddings AlphaEarth de Google (64 números por
píxel y por año) sirven para distinguir tipos de cultivos en dos países:

1. **Italia** — 3 regiones piloto (Pianura Padana, Toscana centrale, Apulia)
   con etiquetas Dynamic World.
2. **Francia** — patches PASTIS-R con 18 cultivos etiquetados por agricultores.
3. **Comparativa cross-region** — qué tan consistentes son las dimensiones
   importantes entre las dos regiones.

## Requisitos para ejecución end-to-end

- `earthengine authenticate` ejecutado y vigente (cuota GEE disponible).
- `data/PASTIS-R/` descomprimido (DATA_S2, ANNOTATIONS, metadata.geojson).
- Dependencias instaladas vía `poetry install --with ml,geo,paper`.

Si EE o PASTIS no están disponibles, las funciones de `ml/ingest/` retornan
DataFrames vacíos con esquema válido y los plots se generan en modo
placeholder (`'Sin datos'`), de forma que el notebook completa la ejecución
sin error.

In [1]:
sample_size = 6_000          # ~2000 px x 3 ROIs Italia (matches 02a budget)
year = 2024
pastis_year = 2019
n_pastis_patches = 10         # patches PASTIS-R a samplear (manejable cuota GEE)
tsne_subsample = 5_000
figures_dir = "paper/figures/us-011"

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import Markdown, display

# Bootstrap del repo root: resolvemos sin importar ml.* primero porque sys.path
# no incluye al repo si el notebook se abre desde notebooks/eda/.
_REPO_BOOTSTRAP = Path.cwd().resolve()
for _candidate in (_REPO_BOOTSTRAP, *_REPO_BOOTSTRAP.parents):
    if (_candidate / "pyproject.toml").is_file():
        _REPO_BOOTSTRAP = _candidate
        break
if str(_REPO_BOOTSTRAP) not in sys.path:
    sys.path.insert(0, str(_REPO_BOOTSTRAP))

from ml.analysis.embeddings import (
    DIM_COLS,
    compare_alphaearth_vs_ndvi,
    correlation_matrix,
    cross_region_consistency,
    qq_test_dims,
    rf_feature_importance,
    temporal_stability,
    tsne_2d,
    umap_2d,
)
from ml.analysis.visualization import (
    correlation_heatmap,
    cross_region_scatter,
    qq_grid,
    tsne_scatter,
    umap_scatter,
)
from ml.ingest.gee_sampler import (
    sample_alphaearth_at_coords,
    sample_alphaearth_roi,
    sample_dynamic_world_at,
)
from ml.ingest.pastis_loader import (
    PASTIS_R_GROUPINGS,
    pastis_patch_coords,
    pastis_pixel_labels,
)
from ml.utils.notebook_setup import find_repo_root
from ml.utils.sampling import stratified_sample

# Polars 1.x: rendering rico HTML en Jupyter + tablas anchas legibles
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_width_chars(180)

# matplotlib inline para que display(fig) y plt.show() rendericen en celda
%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 200

REPO = find_repo_root()
FIGURES = REPO / figures_dir
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE = REPO / "data" / "cache" / "gee"
CACHE.mkdir(parents=True, exist_ok=True)
PASTIS = REPO / "data" / "PASTIS-R"
display(
    Markdown(
        "### Configuración lista\n\n"
        f"- **Repositorio** = `{REPO}`\n"
        f"- **PASTIS disponible** = `{PASTIS.exists()}` ({PASTIS})\n"
        f"- **Carpeta de figuras** = `{FIGURES}`\n"
        f"- **Carpeta de caché** = `{CACHE}`"
    )
)

In [ ]:
# Inicializa Earth Engine (degrada a no-op si la autenticación falla)
from ml.ingest.gee_sampler import init_ee
from ml.utils.notebook_setup import configure_ee_from_env

gee_project, sa_json = configure_ee_from_env(REPO)

ee_available = True
try:
    init_ee(service_account_json=sa_json, project=gee_project)
    display(
        Markdown(
            f"✓ **Earth Engine** inicializado · proyecto=`{gee_project}` · "
            f"autenticación={'service_account' if sa_json else 'ADC'}"
        )
    )
except Exception as exc:
    ee_available = False
    display(
        Markdown(
            f"⚠ **Earth Engine no disponible** · `{exc}`  \n"
            "Las celdas que dependen de GEE degradarán a DataFrame vacío."
        )
    )

## Sección 1 — Italia × Dynamic World

Sampleamos AlphaEarth en las 3 ROIs italianas y joineamos con etiquetas
Dynamic World para tener clases de uso del suelo (crops, trees, built,
water, etc.). Si Earth Engine no está disponible los DataFrames quedan
vacíos y los plots se generan en modo placeholder.

In [ ]:
import time

# Bboxes reducidos (~50x50 km centro de cada region) consistentes con 02a:
# evitan timeout del compute graph server-side y mantienen cuotas razonables.
# Geometria real completa via config/rois.yaml queda para US-006 (ingesta productiva).
ROI_BBOXES: dict[str, list[float]] = {
    "pianura_padana": [10.0, 45.0, 11.0, 45.5],
    "toscana_centrale": [11.0, 43.5, 12.0, 44.0],
    "apulia": [16.0, 41.0, 17.0, 41.5],
}

frames: list[pl.DataFrame] = []
if ee_available:
    import ee  # ya inicializado en cell-3

    n_per_roi = sample_size // len(ROI_BBOXES)
    for idx, (roi_name, bbox) in enumerate(ROI_BBOXES.items(), start=1):
        t0 = time.time()
        geom = ee.Geometry.BBox(*bbox)
        df_roi = sample_alphaearth_roi(
            roi=geom,
            year=year,
            n_pixels=n_per_roi,
            cache_path=CACHE,
            roi_name=roi_name,
        )
        dt = time.time() - t0
        print(f"  [{idx}/{len(ROI_BBOXES)}] {roi_name}: {df_roi.height} px en {dt:.1f}s")
        frames.append(df_roi)
else:
    print("Earth Engine no disponible — se omite el muestreo de AlphaEarth en Italia")

df_italia = pl.concat(frames, how="vertical_relaxed") if frames else pl.DataFrame()
display(Markdown(f"**Muestras de Italia**: `{df_italia.height:,}` filas en {len(ROI_BBOXES)} ROIs"))
if not df_italia.is_empty():
    display(df_italia.head(8))

In [ ]:
# Join con Dynamic World labels
if not df_italia.is_empty():
    coords = df_italia.select(["px_id", "lon", "lat"]).unique(subset=["px_id"])
    dw = sample_dynamic_world_at(coords, year=year, cache_path=CACHE, cache_key="italia")
    df_italia = df_italia.join(dw, on="px_id", how="left")
    display(Markdown("**Distribución de clases Dynamic World en Italia**"))
    display(
        df_italia.group_by("dw_class_name")
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
    )
else:
    display(Markdown("> df_italia vacío — se omite la Sección 1 (modo sin Earth Engine)"))

In [ ]:
# Distribuciones normales por dimension via tests Q-Q
if not df_italia.is_empty():
    qq_italia = qq_test_dims(df_italia)
    display(Markdown("**Estadísticos QQ por dimensión (primeras 8)**"))
    display(qq_italia.head(8))
    fig_qq = qq_grid(qq_italia, out_path=FIGURES / "sec1_qq_italia_dw.png")
    if fig_qq is not None:
        display(fig_qq)
else:
    qq_italia = pl.DataFrame()

In [ ]:
# Matriz de correlacion 64x64 (detecta dimensiones redundantes)
if not df_italia.is_empty():
    corr_italia = correlation_matrix(df_italia, method="pearson")
    fig_corr = correlation_heatmap(
        corr_italia, out_path=FIGURES / "sec1_corr_italia_dw.png", threshold=0.7
    )
    if fig_corr is not None:
        display(fig_corr)
    display(Markdown("**Pares redundantes en Italia (|r| > 0.7)**"))
    display(
        corr_italia.filter(
            (pl.col("abs_corr") > 0.7) & (pl.col("dim_i") != pl.col("dim_j"))
        ).head(10)
    )
else:
    corr_italia = pl.DataFrame()

In [ ]:
# t-SNE y UMAP de los embeddings, coloreados por clase Dynamic World
if not df_italia.is_empty() and "dw_class_name" in df_italia.columns:
    X_tsne, idx_tsne = tsne_2d(df_italia, subsample=tsne_subsample, seed=42)
    labels_tsne = df_italia["dw_class_name"].to_numpy()[idx_tsne]
    fig_tsne = tsne_scatter(
        X_tsne,
        labels_tsne,
        "t-SNE Italia × Dynamic World",
        FIGURES / "sec1_tsne_italia_dw.png",
    )
    if fig_tsne is not None:
        display(fig_tsne)
    X_umap, idx_umap = umap_2d(df_italia, seed=42)
    if X_umap.shape[0] > 0:
        labels_umap = df_italia["dw_class_name"].to_numpy()[idx_umap]
        fig_umap = umap_scatter(
            X_umap,
            labels_umap,
            "UMAP Italia × Dynamic World",
            FIGURES / "sec1_umap_italia_dw.png",
        )
        if fig_umap is not None:
            display(fig_umap)
else:
    display(Markdown("> Se omite t-SNE/UMAP Italia — sin etiquetas Dynamic World"))

In [ ]:
# Random Forest feature importance Italia (que dims ayudan a clasificar mas)
if not df_italia.is_empty() and "dw_class_name" in df_italia.columns:
    df_rf = df_italia.drop_nulls(subset=["dw_class_name", *DIM_COLS])
    rf_italia = rf_feature_importance(
        df_rf.select(DIM_COLS),
        df_rf["dw_class_name"],
        n_estimators=200,
        max_depth=12,
    )
else:
    rf_italia = pl.DataFrame()

if not rf_italia.is_empty():
    display(Markdown("**Top-10 dimensiones AlphaEarth para Italia (importancia Random Forest)**"))
    display(rf_italia.head(10))
    display(Markdown(f"**OOB score Italia**: `{float(rf_italia['oob_score'][0]):.4f}`"))
else:
    display(Markdown("> Se omite Random Forest Italia (DataFrame vacío o sin filas válidas tras dropna)"))

In [ ]:
# Estabilidad temporal 2022-2025 (mismas coords, anios distintos)
stability_frames = []
if ee_available and not df_italia.is_empty():
    coords = df_italia.select(["px_id", "lon", "lat"]).unique(subset=["px_id"]).head(500)
    for y in (2022, 2023, 2024, 2025):
        df_y = sample_alphaearth_at_coords(coords, year=y, cache_path=CACHE, cache_key=f"italia_stab_{y}")
        if "class_name" not in df_y.columns:
            df_y = df_y.with_columns(pl.lit("crops").alias("class_name"))
        stability_frames.append(df_y)
if stability_frames:
    df_stab = pl.concat(stability_frames, how="vertical_relaxed")
    stab = temporal_stability(df_stab)
    display(Markdown("**Estabilidad temporal de AlphaEarth 2022–2025 (cosine similarity por clase)**"))
    display(stab.describe())
    display(stab.head(15))
else:
    display(Markdown("> Se omite el análisis de estabilidad temporal (sin Earth Engine)"))

In [ ]:
# Comparativa visual AlphaEarth vs NDVI clasico
# Tomamos una sub-parcela contigua: 50 pixeles dentro de un bbox de ~500m
# alrededor del centroide, y fetcheamos RGB+NDVI Sentinel-2 reales en esa
# ventana espacial. Sin esto el plot solo renderiza el pseudo-RGB AlphaEarth.
from ml.ingest.gee_sampler import fetch_s2_ndvi_rgb_for_parcel

rgb_arr = None
ndvi_arr = None

if ee_available and not df_italia.is_empty() and not rf_italia.is_empty():
    top3 = rf_italia.head(3)["dim"].to_list()

    # Elegimos un cluster espacial: 50 pixeles mas cercanos al centroide de
    # Pianura Padana (ROI con mas cobertura agricola en Italia).
    df_pp_only = df_italia.filter(pl.col("roi") == "pianura_padana")
    if df_pp_only.is_empty():
        df_pp_only = df_italia  # fallback si filtro vacio
    lon_c = float(df_pp_only["lon"].median())
    lat_c = float(df_pp_only["lat"].median())

    # Bbox ~500 m alrededor del centroide (~0.005 grados) -> parcela compacta
    delta = 0.005
    parcel_bbox = ee.Geometry.Rectangle(
        [lon_c - delta, lat_c - delta, lon_c + delta, lat_c + delta]
    )
    print(f"Descargando RGB + NDVI Sentinel-2 en la parcela @ ({lon_c:.4f}, {lat_c:.4f}) ± {delta}°")
    s2_payload = fetch_s2_ndvi_rgb_for_parcel(
        parcel_geom=parcel_bbox,
        date=f"{year}-06-15",
        cloud_pct_max=20,
        scale=10,
    )
    rgb_arr = s2_payload["rgb"] if s2_payload["rgb"].size > 0 else None
    ndvi_arr = s2_payload["ndvi"] if s2_payload["ndvi"].size > 0 else None
    if rgb_arr is not None:
        print(
            f"  RGB shape={rgb_arr.shape} rango=[{rgb_arr.min():.3f}, {rgb_arr.max():.3f}] "
            f"std_por_banda={[round(float(rgb_arr[..., c].std()), 3) for c in range(3)]}"
        )
    if ndvi_arr is not None:
        print(
            f"  NDVI shape={ndvi_arr.shape} rango=[{ndvi_arr.min():.3f}, {ndvi_arr.max():.3f}] "
            f"media={ndvi_arr.mean():.3f}"
        )

    sample_parcel = df_pp_only.head(50)
    fig_ndvi = compare_alphaearth_vs_ndvi(
        parcel_id=f"pianura_padana @ ({lon_c:.3f}, {lat_c:.3f})",
        df_embeddings=sample_parcel,
        top_dims=top3,
        s2_date=f"{year}-06-15",
        rgb_array=rgb_arr,
        ndvi_array=ndvi_arr,
        out_path=FIGURES / "sec1_alphaearth_vs_ndvi.png",
    )
    if fig_ndvi is not None:
        display(fig_ndvi)
        plt.close(fig_ndvi)  # evita doble render por auto-show de matplotlib inline
else:
    display(Markdown("> Se omite AlphaEarth vs NDVI — rf_italia o df_italia vacíos"))

## Sección 2 — Francia × PASTIS-R

Sampleamos AlphaEarth en las coordenadas reproyectadas (EPSG:2154 → 4326)
de los patches PASTIS-R. Usamos año 2019 porque es el período de cobertura
real del dataset. Filtramos las clases 0 (background) y 19 (void) antes del
muestreo estratificado.

In [ ]:
pastis_meta = REPO / "data" / "PASTIS-R" / "metadata.geojson"
patch_coords = pastis_patch_coords(pastis_meta)
display(Markdown(f"**Patches PASTIS reproyectados (EPSG:2154 → 4326)**: `{patch_coords.height:,}`"))
if patch_coords.is_empty():
    display(Markdown("> PASTIS-R no disponible — la Sección 2 corre en modo placeholder"))
else:
    display(patch_coords.head(8))

In [ ]:
label_frames = []
if not patch_coords.is_empty():
    pastis_root = REPO / "data" / "PASTIS-R"
    for pid in patch_coords.head(n_pastis_patches)["patch_id"].to_list():
        pdf = pastis_pixel_labels(
            pid,
            root=pastis_root,
            sample_per_patch=max(1, sample_size // n_pastis_patches),
            exclude_classes=(0, 19),
        )
        if not pdf.is_empty():
            label_frames.append(pdf)
df_pastis_labels = pl.concat(label_frames, how="vertical_relaxed") if label_frames else pl.DataFrame()
display(Markdown(f"**Píxeles PASTIS muestreados con etiqueta**: `{df_pastis_labels.height:,}`"))
if not df_pastis_labels.is_empty():
    display(Markdown("**Distribución de cultivos PASTIS-R**"))
    display(
        df_pastis_labels.group_by("class_name")
        .agg(pl.len().alias("n"))
        .sort("n", descending=True)
    )

In [ ]:
# Sample AlphaEarth en las coords PASTIS reproyectadas
if not df_pastis_labels.is_empty():
    coords_fr = df_pastis_labels.select(["px_id", "lon", "lat"]).unique(subset=["px_id"])
    df_alphaearth_fr = sample_alphaearth_at_coords(
        coords_fr, year=pastis_year, cache_path=CACHE, cache_key="pastis_fr"
    )
    if not df_alphaearth_fr.is_empty():
        df_francia = df_alphaearth_fr.join(
            df_pastis_labels.select(["px_id", "class_id", "class_name"]),
            on="px_id",
            how="inner",
        )
    else:
        df_francia = pl.DataFrame()
else:
    df_francia = pl.DataFrame()
display(Markdown(f"**Francia (AlphaEarth × PASTIS unidos)**: `{df_francia.height:,}` filas"))
if not df_francia.is_empty():
    display(df_francia.head(8))

In [ ]:
# Correlacion 64x64 Francia
if not df_francia.is_empty():
    corr_fr = correlation_matrix(df_francia)
    fig_corr_fr = correlation_heatmap(
        corr_fr, out_path=FIGURES / "sec2_corr_francia_pastis.png", threshold=0.7
    )
    if fig_corr_fr is not None:
        display(fig_corr_fr)
    display(Markdown("**Pares redundantes en Francia (|r| > 0.7)**"))
    display(
        corr_fr.filter(
            (pl.col("abs_corr") > 0.7) & (pl.col("dim_i") != pl.col("dim_j"))
        ).head(10)
    )
else:
    corr_fr = pl.DataFrame()

In [ ]:
# t-SNE / UMAP Francia coloreados por cultivo
if not df_francia.is_empty() and "class_name" in df_francia.columns:
    X_tsne_fr, idx_fr = tsne_2d(
        df_francia, subsample=min(tsne_subsample, df_francia.height), seed=42
    )
    labels_fr = df_francia["class_name"].to_numpy()[idx_fr]
    fig_tsne_fr = tsne_scatter(
        X_tsne_fr,
        labels_fr,
        "t-SNE Francia × PASTIS-R (18 cultivos)",
        FIGURES / "sec2_tsne_francia_pastis.png",
    )
    if fig_tsne_fr is not None:
        display(fig_tsne_fr)
    X_umap_fr, idx_umap_fr = umap_2d(df_francia, seed=42)
    if X_umap_fr.shape[0] > 0:
        umap_labels_fr = df_francia["class_name"].to_numpy()[idx_umap_fr]
        fig_umap_fr = umap_scatter(
            X_umap_fr,
            umap_labels_fr,
            "UMAP Francia × PASTIS-R",
            FIGURES / "sec2_umap_francia_pastis.png",
        )
        if fig_umap_fr is not None:
            display(fig_umap_fr)
    # Vista agrupada por fenologia (4 clases) - reusa idx_fr de t-SNE
    phen = PASTIS_R_GROUPINGS.get("phenological_cycle", {})
    if phen:
        mapping = pl.DataFrame(
            {"class_id": list(phen.keys()), "phen_group": list(phen.values())},
            schema={"class_id": pl.Int64, "phen_group": pl.Utf8},
        )
        df_phen = df_francia.with_columns(pl.col("class_id").cast(pl.Int64)).join(
            mapping, on="class_id", how="left"
        )
        labels_phen = df_phen["phen_group"].to_numpy()[idx_fr]
        fig_phen = tsne_scatter(
            X_tsne_fr,
            labels_phen,
            "t-SNE Francia × fenología (4 grupos)",
            FIGURES / "sec2_tsne_francia_phenology.png",
        )
        if fig_phen is not None:
            display(fig_phen)
else:
    display(Markdown("> Se omite t-SNE/UMAP Francia — sin df_francia"))

In [ ]:
# RF feature importance Francia
if not df_francia.is_empty() and "class_name" in df_francia.columns:
    df_rf_fr = df_francia.drop_nulls(subset=["class_name", *DIM_COLS])
    rf_francia = rf_feature_importance(
        df_rf_fr.select(DIM_COLS),
        df_rf_fr["class_name"],
        n_estimators=200,
        max_depth=12,
    )
else:
    rf_francia = pl.DataFrame()

if not rf_francia.is_empty():
    display(Markdown("**Top-10 dimensiones AlphaEarth para Francia (importancia Random Forest)**"))
    display(rf_francia.head(10))
    display(Markdown(f"**OOB score Francia**: `{float(rf_francia['oob_score'][0]):.4f}`"))
else:
    display(Markdown("> Se omite Random Forest Francia (DataFrame vacío o sin filas válidas tras dropna)"))

## Sección 3 — Comparativa cross-region

Tabla de consistencia entre las top-10 dimensiones AlphaEarth importantes
para clasificar Italia vs Francia. Si una de las dos ejecuciones está vacía
(EE no disponible o PASTIS no descargado), el plot se omite.

In [ ]:
if not rf_italia.is_empty() and not rf_francia.is_empty():
    consistency = cross_region_consistency(rf_italia, rf_francia, top_k=10)
    display(Markdown("**Tabla de consistencia cross-region Italia ↔ Francia**"))
    display(consistency.head(15))
    n_consistent = int(sum(consistency["consistente_top10"].to_list()))
    status = "✓" if n_consistent >= 5 else "⚠"
    veredicto = (
        "baseline global viable" if n_consistent >= 5 else "requiere regionalización"
    )
    display(
        Markdown(
            f"{status} **Dimensiones consistentes en el top-10**: `{n_consistent}/10` ({veredicto})"
        )
    )
    fig_cross = cross_region_scatter(
        consistency, out_path=FIGURES / "sec3_cross_region_consistency.png"
    )
    if fig_cross is not None:
        display(fig_cross)
else:
    consistency = pl.DataFrame()
    display(Markdown("> Se omite la Sección 3 — rf_italia o rf_francia vacíos"))

## Conclusiones — qué aprendimos de los embeddings AlphaEarth

AlphaEarth es un modelo de Google que comprime cada píxel satelital de 10 m
de la Tierra en un vector de 64 números que resume todo lo que pasó ahí
durante un año (reflectancia, textura, fenología, contexto). Aquí evaluamos
si esos 64 números sirven para distinguir tipos de cultivos en dos países muy
diferentes — **Italia** (3 regiones piloto con etiquetas Dynamic World, 6,000
píxeles) y **Francia** (10 patches PASTIS-R con 13 cultivos etiquetados por
agricultores, 5,000 píxeles).

### Lo que funcionó

1. **Los embeddings separan cultivos sin necesidad de feature engineering
   manual.** En los gráficos t-SNE y UMAP se ven clusters claros: campos
   agrícolas se agrupan separados de zonas construidas, agua y bosque. No
   tuvimos que calcular índices (NDVI, EVI, NDWI) — los 64 números ya
   capturaron esa información.

2. **Un clasificador simple sobre los 64 embeddings funciona bien.** Un
   Random Forest básico logra:
   - **OOB score 0.888 en Italia** (8 clases Dynamic World)
   - **OOB score 0.831 en Francia** (13 cultivos PASTIS-R)

   Sin imágenes, sin series temporales, sin DINOv3 — solo el embedding anual.

3. **AlphaEarth carga más señal que el NDVI clásico.** El panel comparativo
   muestra el RGB Sentinel-2 real al lado del "pseudo-RGB" construido con las
   3 dimensiones AlphaEarth más informativas. El pseudo-RGB resalta variación
   estructural (parcelas, caminos) que un solo índice escalar como NDVI
   aplana.

### El hallazgo importante: los embeddings se especializan por región

Cuando comparamos cuáles dimensiones son las más útiles para clasificar en
Italia versus en Francia, **solo 1 de las 10 mejores coincide entre ambas**
(`dim_40`). El resto del top-10 cambia totalmente:

- **Italia** se apoya en `dim_30, dim_36, dim_16, dim_07, dim_63` — dimensiones
  que casi no aparecen en el top de Francia.
- **Francia** se apoya en `dim_10, dim_55, dim_05, dim_32, dim_60` — dimensiones
  que casi no aparecen en el top de Italia.

Esto significa que AlphaEarth aprendió representaciones que **dependen del
contexto regional**: el clima mediterráneo de Apulia, el patrón parcelario
de la Pianura Padana y las rotaciones cereal/colza francesas usan partes
distintas del embedding.

**Consecuencia práctica**: entrenar UN solo clasificador global para Italia
y Francia probablemente rinda menos que entrenar **un clasificador por
región**. Más adelante, cuando construyamos el modelo baseline para predecir
cultivos, vamos a entrenarlos separados.

### Hallazgo: redundancia entre dimensiones

La matriz de correlación 64×64 detecta varios pares de dimensiones con
correlación `|r| > 0.7`. Eso significa que algunas dimensiones cargan
información parecida y podríamos colapsarlas con PCA antes de alimentar
XGBoost. No es bloqueante pero reduce ruido en el modelo final.

### Estabilidad temporal: AlphaEarth es estable entre años

Probamos si los embeddings cambian mucho de un año a otro en las **mismas
coordenadas** (2022, 2023, 2024, 2025). El cosine similarity promedio entre
años consecutivos en 500 píxeles aleatorios fue:

| Comparación | Similaridad promedio | Mediana |
|-------------|----------------------|---------|
| 2022 vs 2023 | 0.952 | 0.967 |
| 2023 vs 2024 | 0.953 | 0.975 |
| 2024 vs 2025 | 0.954 | 0.976 |

Mediana > 0.97 en todos los pares. Esto significa que para parcelas con uso
del suelo constante, **podemos entrenar con un solo año y predecir en años
cercanos sin re-extraer features**. Ahorra cuota GEE y tiempo de cómputo.

### Sobre los grupos fenológicos en Francia

El t-SNE coloreado por ciclo fenológico muestra solo 3 grupos
(`winter_crops`, `spring_summer_crops`, `other`) aunque PASTIS-R define 4.
Los cultivos **permanentes** (viñedos, frutales, olivares) no aparecieron en
el muestreo porque solo procesamos 10 patches PASTIS y son una clase
minoritaria a esa escala. En un muestreo más amplio aparecerían — no es un
bug, es muestreo pequeño.

### Lo que sigue

- **Modelo baseline regional**: entrenar XGBoost por separado para Italia y
  Francia, usando las dims más informativas de cada región.
- **Sanity check NDVI**: comparar XGBoost+AlphaEarth contra XGBoost+NDVI para
  cuantificar la ganancia del Foundation Model sobre un índice clásico.
- **PCA opcional**: colapsar los pares con `|r| > 0.7` antes del clasificador
  para reducir ruido.
- **Comparación con DINOv3**: evaluar si vale la pena pagar el costo
  computacional de un extractor más pesado o si AlphaEarth basta.